<a href="https://colab.research.google.com/github/fethinho/AI-Trader/blob/main/salma_swing_scanner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

The `bist_tickers.csv` file has been updated to remove the specified problematic symbols. You can now re-run the main scanning script to use the cleaned list of tickers.

In [22]:
import re
import requests
import pandas as pd
import numpy as np
import yfinance as yf

# ----------------- Sabitler ----------------- #

BOT_TOKEN = "8460458441:AAG81QuP0nDc0AT5SYON63bus2d0udab0Iw"
CHAT_ID = "314746106"
TICKER_FILE = "bist_tickers.csv"

TIMEFRAMES = {
    "15m": {"period": "5d",  "interval": "15m"},
    "1h":  {"period": "1mo", "interval": "1h"},
    "4h":  {"period": "3mo", "interval": "1h"},  # 4h için 1h bazından aggregate
    "1d":  {"period": "6mo", "interval": "1d"},
}

MIN_SCORE = 60              # minimum güçlü sinyal eşiği
MAX_ROWS_PER_MESSAGE = 25   # her mesajda max satır


# ----------------- Yardımcı fonksiyonlar ----------------- #

def load_symbols():
    with open(TICKER_FILE, "r", encoding="utf-8") as f:
        raw = f.read()
    symbols = re.findall(r"[A-Z0-9\.]+\.IS", raw.upper())
    return list(dict.fromkeys(symbols))

def wma(series, length):
    weights = np.arange(1, length + 1)
    return series.rolling(length).apply(
        lambda x: np.dot(x, weights) / weights.sum(), raw=True
    )

def ema(series, length):
    return series.ewm(span=length, adjust=False).mean()

def rsi(series, length=14):
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.rolling(length).mean()
    avg_loss = loss.rolling(length).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    return 100 - (100 / (1 + rs))

def atr(df, length=14):
    high = pd.to_numeric(df["High"], errors="coerce")
    low = pd.to_numeric(df["Low"], errors="coerce")
    close = pd.to_numeric(df["Close"], errors="coerce")

    tr = pd.concat([
        high - low,
        (high - close.shift(1)).abs(),
        (low - close.shift(1)).abs()
    ], axis=1).max(axis=1)

    return tr.rolling(length).mean()

def calc_salma(df, length=10, smooth=2, sd_len=3, multi=0.3):
    close = pd.to_numeric(df["Close"], errors="coerce")
    baseline = wma(close, sd_len)
    dev = multi * close.rolling(sd_len).std()
    upper = baseline + dev
    lower = baseline - dev
    cprice = np.where(close > upper, upper, np.where(close < lower, lower, close))
    cprice = pd.Series(cprice, index=df.index)
    salma = wma(wma(cprice, length), smooth)
    return salma

def normalize_df(df):
    if df is None or df.empty:
        return None
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    needed = ["Open", "High", "Low", "Close", "Volume"]
    for col in needed:
        if col not in df.columns:
            return None
        df[col] = pd.to_numeric(df[col], errors="coerce")
    df = df.dropna(subset=needed).copy()
    if len(df) < 30:
        return None
    return df

def get_salma_signal(df):
    df = normalize_df(df)
    if df is None or len(df) < 30:
        return None

    salma = calc_salma(df)
    rema_up = salma > salma.shift(1)

    swing_up = (rema_up == True) & (rema_up.shift(1) != True)
    swing_dn = (rema_up == False) & (rema_up.shift(1) == True)

    idx = -2 if len(df) >= 2 else -1
    if bool(swing_up.iloc[idx]):
        return "AL"
    if bool(swing_dn.iloc[idx]):
        return "SAT"
    return None

def send_telegram(message):
    url = f"https://api.telegram.org/bot{BOT_TOKEN}/sendMessage"
    requests.post(
        url,
        data={"chat_id": CHAT_ID, "text": message},
        timeout=20
    )

def fetch_data(symbol, interval, period):
    return yf.download(
        symbol,
        interval=interval,
        period=period,
        auto_adjust=True,
        progress=False,
        threads=False
    )

def aggregate_to_4h(df_1h):
    df_1h = normalize_df(df_1h)
    if df_1h is None:
        return None

    df_4h = df_1h[["Open", "High", "Low", "Close", "Volume"]].resample("4h").agg({
        "Open": "first",
        "High": "max",
        "Low": "min",
        "Close": "last",
        "Volume": "sum"
    }).dropna()

    return df_4h if len(df_4h) >= 30 else None

def get_tf_dataframe(symbol, tf):
    cfg = TIMEFRAMES[tf]
    if tf == "4h":
        base_df = fetch_data(symbol, "1h", "3mo")
        return aggregate_to_4h(base_df)
    return normalize_df(fetch_data(symbol, cfg["interval"], cfg["period"]))

def higher_tf_confirm(symbol):
    daily_df = get_tf_dataframe(symbol, "1d")
    if daily_df is None or len(daily_df) < 60:
        return False
    close = daily_df["Close"]
    ema50_d = ema(close, 50)
    return bool(close.iloc[-2] > ema50_d.iloc[-2])


# ----------------- Skor fonksiyonları ----------------- #

def score_dataframe(df, higher_tf_ok=False):
    df = normalize_df(df)
    if df is None or len(df) < 60:
        return None

    signal = get_salma_signal(df)
    if signal != "AL":
        return None

    close = df["Close"]
    volume = df["Volume"]

    ema50_s = ema(close, 50)
    ema200_s = ema(close, 200)
    rsi14 = rsi(close, 14)
    atr14 = atr(df, 14)
    atr_ma20 = atr14.rolling(20).mean()
    vol_ma20 = volume.rolling(20).mean()
    rvol20 = volume / vol_ma20.replace(0, np.nan)

    idx = -2 if len(df) >= 2 else -1

    score = 40
    reasons = ["SALMA_AL"]

    rvol_value = rvol20.iloc[idx]
    rsi_value = rsi14.iloc[idx]
    atr_value = atr14.iloc[idx]
    atr_avg_value = atr_ma20.iloc[idx]
    close_value = close.iloc[idx]
    ema50_value = ema50_s.iloc[idx]
    ema200_value = ema200_s.iloc[idx]

    if pd.notna(rvol_value):
        if rvol_value >= 2.0:
            score += 25
            reasons.append("RVOL2+")
        elif rvol_value >= 1.5:
            score += 18
            reasons.append("RVOL1.5+")
        elif rvol_value >= 1.2:
            score += 10
            reasons.append("RVOL1.2+")

    if pd.notna(close_value) and pd.notna(ema50_value) and close_value > ema50_value:
        score += 10
        reasons.append("EMA50_USTU")

    if pd.notna(ema50_value) and pd.notna(ema200_value) and ema50_value > ema200_value:
        score += 10
        reasons.append("EMA50_EMA200_USTU")

    if pd.notna(rsi_value):
        if rsi_value >= 60:
            score += 10
            reasons.append("RSI60+")
        elif rsi_value >= 55:
            score += 7
            reasons.append("RSI55+")

    if pd.notna(atr_value) and pd.notna(atr_avg_value) and atr_value > atr_avg_value:
        score += 8
        reasons.append("ATR_GUCLU")

    if higher_tf_ok:
        score += 10
        reasons.append("UST_TF_ONAY")

    return {
        "score": int(score),
        "price": float(close.iloc[-1]),
        "rvol": float(rvol_value) if pd.notna(rvol_value) else None,
        "rsi": float(rsi_value) if pd.notna(rsi_value) else None,
        "reasons": reasons
    }

def scan_symbol(symbol):
    results = []
    htf_ok = higher_tf_confirm(symbol)

    for tf in ["15m", "1h", "4h", "1d"]:
        try:
            df = get_tf_dataframe(symbol, tf)
            scored = score_dataframe(df, higher_tf_ok=htf_ok if tf != "1d" else False)
            if scored and scored["score"] >= MIN_SCORE:
                results.append({
                    "sym": symbol,
                    "tf": tf,
                    "price": scored["price"],
                    "score": scored["score"],
                    "rvol": scored["rvol"],
                    "rsi": scored["rsi"],
                    "reasons": scored["reasons"]
                })
        except Exception:
            continue

    return results


# ----------------- Format fonksiyonları ----------------- #

def format_table(rows, title):
    if not rows:
        return ""

    header = f"{title}\n"
    sub_header = f"{'TF':<4} {'Sembol':<10} {'Skor':>5} {'RVOL':>6} {'RSI':>6} {'Fiyat':>10}"
    sep = "-" * len(sub_header)

    lines = [header, sub_header, sep]

    for r in rows:
        rvol_txt = f"{r['rvol']:.2f}" if r["rvol"] is not None else "-"
        rsi_txt  = f"{r['rsi']:.1f}" if r["rsi"] is not None else "-"
        lines.append(
            f"{r['tf']:<4} "
            f"{r['sym']:<10} "
            f"{r['score']:>5} "
            f"{rvol_txt:>6} "
            f"{rsi_txt:>6} "
            f"{r['price']:>10.2f}"
        )
    return "\n".join(lines)

def format_reason_table(rows):
    top = sorted(
        rows,
        key=lambda x: (x["score"], x["rvol"] if x["rvol"] is not None else 0),
        reverse=True
    )[:10]

    header = f"{'TF':<4} {'Sembol':<10} {'Skor':>5} {'RVOL':>6} {'RSI':>6}  Neden"
    sep    = "-" * 80
    lines  = [header, sep]

    for r in top:
        rvol_txt = f"{r['rvol']:.2f}" if r["rvol"] is not None else "-"
        rsi_txt  = f"{r['rsi']:.1f}" if r["rsi"] is not None else "-"
        reasons  = ",".join(r["reasons"])
        lines.append(
            f"{r['tf']:<4} "
            f"{r['sym']:<10} "
            f"{r['score']:>5} "
            f"{rvol_txt:>6} "
            f"{rsi_txt:>6}  "
            f"{reasons}"
        )

    return "\n".join(lines)


# ----------------- Mesaj göndericiler ----------------- #

def send_intraday(all_hits):
    intraday = [r for r in all_hits if r["tf"] in ("15m", "1h")]
    if not intraday:
        return

    intraday = sorted(
        intraday,
        key=lambda x: (x["tf"], -x["score"], -(x["rvol"] or 0))
    )[:MAX_ROWS_PER_MESSAGE]

    text = format_table(intraday, "SALMA + SWING GÜÇLÜ AL\n[İntraday 15m / 1h]")
    send_telegram(text)

def send_swing(all_hits):
    swing = [r for r in all_hits if r["tf"] in ("4h", "1d")]
    if not swing:
        return

    swing = sorted(
        swing,
        key=lambda x: (x["tf"], -x["score"], -(x["rvol"] or 0))
    )[:MAX_ROWS_PER_MESSAGE]

    text = format_table(swing, "SALMA + SWING GÜÇLÜ AL\n[4h / Günlük]")
    send_telegram(text)


# ----------------- main ----------------- #

def main():
    symbols = load_symbols()
    all_hits = []

    for symbol in symbols:
        hits = scan_symbol(symbol)
        all_hits.extend(hits)

    if not all_hits:
        return

    # 1) İntraday tablo
    send_intraday(all_hits)

    # 2) Swing tablo
    send_swing(all_hits)

    # 3) Detay tablo (ilk 10) - monospace code block
    reason_table = format_reason_table(all_hits)
    detail_message = (
        "SALMA + SWING GÜÇLÜ AL\n"
        "[Detay - ilk 10]\n\n"
        "```" + "\n" + reason_table + "\n" + "```"
    )
    send_telegram(detail_message)

main()